In [76]:
import os
import pandas as pd

from dotenv import load_dotenv
from pymongo import MongoClient
from collections import defaultdict

load_dotenv("/home/shbong/Project2-ROS2-VLA/.env")

client = MongoClient(
    host="127.0.0.1",
    port=27018,  # SSH 터널의 로컬 포트
    username=os.environ["MONGO_ROOT_USER"],
    password=os.environ["MONGO_ROOT_PASSWORD"],
    authSource=os.getenv("MONGO_AUTH_DATABASE", "admin"),
    serverSelectionTimeoutMS=5000,
)

client.admin.command("ping")
db = client[os.environ["MONGO_DATABASE"]]

print("MongoDB 연결 성공")
print(db.list_collection_names())


MongoDB 연결 성공
['commands', 'component_executions', 'kit_executions']


### 품목 이름 확인

In [77]:
db.component_executions.distinct("class_name")

['마스크', '분유', '샴푸리필', '수세미', '양갱', '여행용티슈', '일회용숟가락', '컵라면', '햄']

### task list 확인

In [78]:
# 실제 테스트 시간으로 수정
start_kst = pd.Timestamp("2026-09-08 15:20", tz="Asia/Seoul")
end_kst   = pd.Timestamp("2026-09-08 17:20", tz="Asia/Seoul")

start_utc = start_kst.tz_convert("UTC").to_pydatetime()
end_utc = end_kst.tz_convert("UTC").to_pydatetime()

kits = list(
    db.kit_executions.find({
        "started_at": {
            "$gte": start_utc,
            "$lt": end_utc,
        }
    }).sort("started_at", 1)
)

task_ids = [doc["task_id"] for doc in kits]

commands = {
    doc["task_id"]: doc
    for doc in db.commands.find({"task_id": {"$in": task_ids}})
}

components_by_task = defaultdict(list)

for doc in db.component_executions.find({
    "task_id": {"$in": task_ids}
}):
    components_by_task[doc["task_id"]].append(doc)


In [79]:
rows = []

for kit in kits:
    task_id = kit["task_id"]
    command = commands.get(task_id, {})
    components = components_by_task.get(task_id, [])

    # SKIPPED는 실제 파지 동작이 수행되지 않은 Component
    executed = [
        component
        for component in components
        if component.get("status") != "SKIPPED"
    ]

    rows.append({
        "task_id": task_id,
        "started_at": kit.get("started_at"),
        "ended_at": kit.get("ended_at"),
        "raw_text": command.get("raw_text"),
        "validation_success": (
            command.get("validation", {}).get("success")
        ),
        "command_error": (
            command.get("validation", {}).get("error_code")
        ),
        "kit_type": kit.get("kit_type"),
        "task_status": kit.get("status"),
        "task_error": kit.get("error", {}).get("code"),
        "component_total": kit.get("component_total"),
        "saved_component_count": len(components),
        "executed_component_count": len(executed),
        "classes": ", ".join(
            component.get("class_name", "")
            for component in components
        ),
        "component_statuses": ", ".join(
            component.get("status", "")
            for component in components
        ),
    })

audit = pd.DataFrame(rows).sort_values("started_at")
audit


,task_id,started_at,ended_at,raw_text,validation_success,command_error,kit_type,task_status,task_error,component_total,saved_component_count,executed_component_count,classes,component_statuses
0,TASK-20260908T062225060390Z,2026-09-08 06:22:25.075,2026-09-08 06:26:43.086,응급키트 1번 만들어줘,True,NaN,응급키트1번,FAILED,inspection_mismatch,5,5,5,"수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS"
1,TASK-20260908T062653244581Z,2026-09-08 06:26:53.245,2026-09-08 06:30:28.491,응급키트 1번 만들어줘,True,NaN,응급키트1번,FAILED,inspection_mismatch,5,5,5,"수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, FAILED, SUCCESS, SUCCESS, SUCCESS"
2,TASK-20260908T063038657848Z,2026-09-08 06:30:38.660,2026-09-08 06:30:54.040,샴푸 리필이랑 응급키트 1번 만들어줘,True,NaN,,FAILED,invalid_command,0,0,0,,
3,TASK-20260908T063104191011Z,2026-09-08 06:31:04.192,2026-09-08 06:35:25.792,응급키트 1번 만들어줘,True,NaN,응급키트1번,FAILED,inspection_mismatch,6,6,6,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, SUCCESS, SU..."
4,TASK-20260908T063535978796Z,2026-09-08 06:35:35.979,2026-09-08 06:39:11.207,응급키트 1번 만들어,True,NaN,응급키트1번,FAILED,inspection_response_invalid,6,6,6,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, FAILED, FAILED"
5,TASK-20260908T063921355753Z,2026-09-08 06:39:21.356,2026-09-08 06:43:54.901,응급키트 1번 만들어,True,NaN,응급키트1번,FAILED,inspection_mismatch,6,6,6,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, SUCCESS, SU..."
6,TASK-20260908T064405082255Z,2026-09-08 06:44:05.083,2026-09-08 06:44:35.310,,False,wakeword_timeout,,FAILED,wakeword_timeout,0,0,0,,
7,TASK-20260908T064445519071Z,2026-09-08 06:44:45.520,2026-09-08 06:45:15.831,,False,wakeword_timeout,,FAILED,wakeword_timeout,0,0,0,,
8,TASK-20260908T064526000816Z,2026-09-08 06:45:26.002,2026-09-08 06:50:41.505,엘로우 로키 응급키트 2번 만들어,True,NaN,응급키트2번,FAILED,inspection_mismatch,6,6,6,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS, S..."
9,TASK-20260908T065051667419Z,2026-09-08 06:50:51.670,2026-09-08 06:52:41.968,응급키트 2번 만들어,True,NaN,응급키트2번,FAILED,recovery_failed,6,6,2,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, FAILED, SKIPPED, SKIPPED, SKIPPED, SK..."


In [80]:
### 오류 작업과 실제 실행 작업 분리

eligible = audit[
    audit["validation_success"].eq(True)
    & audit["executed_component_count"].gt(0)
].copy()

excluded = audit.drop(eligible.index)

print("전체 DB 작업:", len(audit))
print("실제 동작 후보:", len(eligible))
print("사전 오류/미실행:", len(excluded))


전체 DB 작업: 39
실제 동작 후보: 22
사전 오류/미실행: 17


#### 제외 항목

In [81]:
excluded[
    [
        "task_id",
        "started_at",
        "raw_text",
        "command_error",
        "task_error",
        "task_status",
        "executed_component_count",
    ]
]


,task_id,started_at,raw_text,command_error,task_error,task_status,executed_component_count
2,TASK-20260908T063038657848Z,2026-09-08 06:30:38.660,샴푸 리필이랑 응급키트 1번 만들어줘,NaN,invalid_command,FAILED,0
6,TASK-20260908T064405082255Z,2026-09-08 06:44:05.083,,wakeword_timeout,wakeword_timeout,FAILED,0
7,TASK-20260908T064445519071Z,2026-09-08 06:44:45.520,,wakeword_timeout,wakeword_timeout,FAILED,0
11,TASK-20260908T070342642831Z,2026-09-08 07:03:42.675,응급키트 2번 만들어,NaN,NaN,RUNNING,0
15,TASK-20260908T071758184885Z,2026-09-08 07:17:58.188,NaN,NaN,NaN,RUNNING,0
17,TASK-20260908T072534474854Z,2026-09-08 07:25:34.476,취사키트 2번 만들어 1번 취사키트 1번,NaN,invalid_command,FAILED,0
18,TASK-20260908T072611418018Z,2026-09-08 07:26:11.419,NaN,NaN,invalid_command,FAILED,0
20,TASK-20260908T073123593534Z,2026-09-08 07:31:23.594,NaN,NaN,invalid_command,FAILED,0
23,TASK-20260908T074235775619Z,2026-09-08 07:42:35.776,,wakeword_timeout,wakeword_timeout,FAILED,0
25,TASK-20260908T074827008281Z,2026-09-08 07:48:27.009,,wakeword_timeout,wakeword_timeout,FAILED,0


#### 실제 실행 후보

In [82]:
eligible[
    [
        "task_id",
        "started_at",
        "raw_text",
        "kit_type",
        "task_status",
        "classes",
        "component_statuses",
    ]
].reset_index(drop=True)


,task_id,started_at,raw_text,kit_type,task_status,classes,component_statuses
0,TASK-20260908T062225060390Z,2026-09-08 06:22:25.075,응급키트 1번 만들어줘,응급키트1번,FAILED,"수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS"
1,TASK-20260908T062653244581Z,2026-09-08 06:26:53.245,응급키트 1번 만들어줘,응급키트1번,FAILED,"수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, FAILED, SUCCESS, SUCCESS, SUCCESS"
2,TASK-20260908T063104191011Z,2026-09-08 06:31:04.192,응급키트 1번 만들어줘,응급키트1번,FAILED,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, SUCCESS, SU..."
3,TASK-20260908T063535978796Z,2026-09-08 06:35:35.979,응급키트 1번 만들어,응급키트1번,FAILED,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, FAILED, FAILED"
4,TASK-20260908T063921355753Z,2026-09-08 06:39:21.356,응급키트 1번 만들어,응급키트1번,FAILED,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, SUCCESS, SU..."
5,TASK-20260908T064526000816Z,2026-09-08 06:45:26.002,엘로우 로키 응급키트 2번 만들어,응급키트2번,FAILED,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS, S..."
6,TASK-20260908T065051667419Z,2026-09-08 06:50:51.670,응급키트 2번 만들어,응급키트2번,FAILED,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, FAILED, SKIPPED, SKIPPED, SKIPPED, SK..."
7,TASK-20260908T065842478845Z,2026-09-08 06:58:42.494,응급기트 2번 만들어,응급키트2번,FAILED,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","FAILED, SKIPPED, SKIPPED, SKIPPED, SKIPPED, SK..."
8,TASK-20260908T070429730427Z,2026-09-08 07:04:29.747,응급키트 2번 만들어,응급키트2번,FAILED,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS, S..."
9,TASK-20260908T070930692553Z,2026-09-08 07:09:30.694,응급키트 2번 만들어,응급키트2번,FAILED,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, FAILED, SUCCESS, FAILED, SUC..."


In [83]:
selected = eligible.copy()

selected

,task_id,started_at,ended_at,raw_text,validation_success,command_error,kit_type,task_status,task_error,component_total,saved_component_count,executed_component_count,classes,component_statuses
0,TASK-20260908T062225060390Z,2026-09-08 06:22:25.075,2026-09-08 06:26:43.086,응급키트 1번 만들어줘,True,NaN,응급키트1번,FAILED,inspection_mismatch,5,5,5,"수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS"
1,TASK-20260908T062653244581Z,2026-09-08 06:26:53.245,2026-09-08 06:30:28.491,응급키트 1번 만들어줘,True,NaN,응급키트1번,FAILED,inspection_mismatch,5,5,5,"수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, FAILED, SUCCESS, SUCCESS, SUCCESS"
3,TASK-20260908T063104191011Z,2026-09-08 06:31:04.192,2026-09-08 06:35:25.792,응급키트 1번 만들어줘,True,NaN,응급키트1번,FAILED,inspection_mismatch,6,6,6,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, SUCCESS, SU..."
4,TASK-20260908T063535978796Z,2026-09-08 06:35:35.979,2026-09-08 06:39:11.207,응급키트 1번 만들어,True,NaN,응급키트1번,FAILED,inspection_response_invalid,6,6,6,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, FAILED, FAILED"
5,TASK-20260908T063921355753Z,2026-09-08 06:39:21.356,2026-09-08 06:43:54.901,응급키트 1번 만들어,True,NaN,응급키트1번,FAILED,inspection_mismatch,6,6,6,"샴푸리필, 수세미, 양갱, 여행용티슈, 컵라면, 햄","SUCCESS, SUCCESS, FAILED, SUCCESS, SUCCESS, SU..."
8,TASK-20260908T064526000816Z,2026-09-08 06:45:26.002,2026-09-08 06:50:41.505,엘로우 로키 응급키트 2번 만들어,True,NaN,응급키트2번,FAILED,inspection_mismatch,6,6,6,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS, S..."
9,TASK-20260908T065051667419Z,2026-09-08 06:50:51.670,2026-09-08 06:52:41.968,응급키트 2번 만들어,True,NaN,응급키트2번,FAILED,recovery_failed,6,6,2,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, FAILED, SKIPPED, SKIPPED, SKIPPED, SK..."
10,TASK-20260908T065842478845Z,2026-09-08 06:58:42.494,2026-09-08 06:59:43.353,응급기트 2번 만들어,True,NaN,응급키트2번,FAILED,recovery_failed,6,6,1,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","FAILED, SKIPPED, SKIPPED, SKIPPED, SKIPPED, SK..."
12,TASK-20260908T070429730427Z,2026-09-08 07:04:29.747,2026-09-08 07:09:20.519,응급키트 2번 만들어,True,NaN,응급키트2번,FAILED,inspection_mismatch,6,6,6,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS, S..."
13,TASK-20260908T070930692553Z,2026-09-08 07:09:30.694,2026-09-08 07:13:32.218,응급키트 2번 만들어,True,NaN,응급키트2번,FAILED,inspection_mismatch,6,6,6,"마스크, 분유, 분유, 샴푸리필, 여행용티슈, 일회용숟가락","SUCCESS, SUCCESS, FAILED, SUCCESS, FAILED, SUC..."


### 음성: 사전 정의 명령에 대한 STT 문장 또는 키워드 인식률

In [84]:
import re

KEYWORDS = [
    "응급키트 1번",
    "응급키트 2번",
    "취사키트 1번",
    "취사키트 2번",
]

def normalize_text(value):
    """공백과 문장부호를 제거해 비교한다."""
    return re.sub(
        r"[^0-9a-zA-Z가-힣]+",
        "",
        str(value or "").lower(),
    )

def extract_keywords(raw_text):
    normalized = normalize_text(raw_text)

    return [
        keyword
        for keyword in KEYWORDS
        if normalize_text(keyword) in normalized
    ]

In [85]:
### 1. raw_text 키워드 포함 여부

keyword_eval =  selected[
    ["task_id", "started_at", "raw_text"]
].copy()

keyword_eval["detected_keywords"] = (
    keyword_eval["raw_text"].apply(extract_keywords)
)

keyword_eval["detected_keyword"] = (
    keyword_eval["detected_keywords"]
    .apply(lambda values: values[0] if len(values) == 1 else None)
)

keyword_eval["keyword_found"] = (
    keyword_eval["detected_keywords"].apply(bool)
)

keyword_eval["multiple_keywords"] = (
    keyword_eval["detected_keywords"].apply(
        lambda values: len(values) > 1
    )
)

keyword_eval

,task_id,started_at,raw_text,detected_keywords,detected_keyword,keyword_found,multiple_keywords
0,TASK-20260908T062225060390Z,2026-09-08 06:22:25.075,응급키트 1번 만들어줘,[응급키트 1번],응급키트 1번,True,False
1,TASK-20260908T062653244581Z,2026-09-08 06:26:53.245,응급키트 1번 만들어줘,[응급키트 1번],응급키트 1번,True,False
3,TASK-20260908T063104191011Z,2026-09-08 06:31:04.192,응급키트 1번 만들어줘,[응급키트 1번],응급키트 1번,True,False
4,TASK-20260908T063535978796Z,2026-09-08 06:35:35.979,응급키트 1번 만들어,[응급키트 1번],응급키트 1번,True,False
5,TASK-20260908T063921355753Z,2026-09-08 06:39:21.356,응급키트 1번 만들어,[응급키트 1번],응급키트 1번,True,False
8,TASK-20260908T064526000816Z,2026-09-08 06:45:26.002,엘로우 로키 응급키트 2번 만들어,[응급키트 2번],응급키트 2번,True,False
9,TASK-20260908T065051667419Z,2026-09-08 06:50:51.670,응급키트 2번 만들어,[응급키트 2번],응급키트 2번,True,False
10,TASK-20260908T065842478845Z,2026-09-08 06:58:42.494,응급기트 2번 만들어,[],NaN,False,False
12,TASK-20260908T070429730427Z,2026-09-08 07:04:29.747,응급키트 2번 만들어,[응급키트 2번],응급키트 2번,True,False
13,TASK-20260908T070930692553Z,2026-09-08 07:09:30.694,응급키트 2번 만들어,[응급키트 2번],응급키트 2번,True,False


In [86]:
### 2. 전체 키워드 추출률

total_count = len(keyword_eval)
found_count = int(keyword_eval["keyword_found"].sum())
extraction_rate = (
    found_count / total_count * 100
    if total_count else 0
)

print(f"키워드 추출 성공: {found_count}/{total_count}")
print(f"키워드 추출률: {extraction_rate:.2f}%")


키워드 추출 성공: 20/22
키워드 추출률: 90.91%


In [87]:
### 3. 키워드별 발견 횟수

keyword_counts = (
    keyword_eval
    .explode("detected_keywords")
    .dropna(subset=["detected_keywords"])
    .groupby("detected_keywords")
    .size()
    .reindex(KEYWORDS, fill_value=0)
    .rename("detected_count")
    .reset_index()
    .rename(columns={"detected_keywords": "keyword"})
)

keyword_counts["percentage_of_all_trials"] = (
    keyword_counts["detected_count"] / total_count * 100
).round(2)

keyword_counts


,keyword,detected_count,percentage_of_all_trials
0,응급키트 1번,5,22.73
1,응급키트 2번,5,22.73
2,취사키트 1번,5,22.73
3,취사키트 2번,5,22.73


In [88]:
### 4. 인식 실패 데이터 확인

keyword_eval.loc[
    ~keyword_eval["keyword_found"],
    ["task_id", "started_at", "raw_text"],
]

,task_id,started_at,raw_text
10,TASK-20260908T065842478845Z,2026-09-08 06:58:42.494,응급기트 2번 만들어
37,TASK-20260908T081351275574Z,2026-09-08 08:13:51.276,시사 킬투 2번 만들어


### 명령 변환: 필수 필드와 형식을 충족한 JSON 유효 응답률

In [89]:
import json
import pandas as pd

selected_task_ids = selected["task_id"].dropna().tolist()

command_docs = list(
    db.commands.find(
        {"task_id": {"$in": selected_task_ids}},
        {
            "_id": 0,
            "task_id": 1,
            "raw_text": 1,
            "command": 1,
            "validation": 1,
            "created_at": 1,
        },
    )
)

In [90]:
def validate_command(command):
    errors = []

    # MongoDB에 문자열 JSON으로 저장된 경우도 처리
    if isinstance(command, str):
        try:
            command = json.loads(command)
        except json.JSONDecodeError:
            return False, ["JSON 파싱 실패"]

    if not isinstance(command, dict):
        return False, ["command가 JSON object가 아님"]

    kit_type = command.get("kit_type")

    if not isinstance(kit_type, str) or not kit_type.strip():
        errors.append("kit_type이 비어 있거나 문자열이 아님")

    items = command.get("items")

    if not isinstance(items, list) or not items:
        errors.append("items가 비어 있거나 배열이 아님")
    else:
        for index, item in enumerate(items):
            if not isinstance(item, dict):
                errors.append(f"items[{index}]가 object가 아님")
                continue

            name = item.get("name")
            qty = item.get("qty")

            if not isinstance(name, str) or not name.strip():
                errors.append(f"items[{index}].name 오류")

            # bool은 Python에서 int의 하위 타입이므로 별도 제외
            if (
                isinstance(qty, bool)
                or not isinstance(qty, int)
                or qty <= 0
            ):
                errors.append(
                    f"items[{index}].qty는 양의 정수여야 함"
                )

    return len(errors) == 0, errors


In [91]:
rows = []

for doc in command_docs:
    validation = doc.get("validation") or {}

    # 명령 변환 자체가 성공한 항목만 분모에 포함
    if validation.get("success") is not True:
        continue

    is_valid, errors = validate_command(doc.get("command"))

    rows.append({
        "task_id": doc.get("task_id"),
        "raw_text": doc.get("raw_text"),
        "command": doc.get("command"),
        "json_valid": is_valid,
        "validation_errors": errors,
        "created_at": doc.get("created_at"),
    })

command_eval = pd.DataFrame(rows)
command_eval


,task_id,raw_text,command,json_valid,validation_errors,created_at
0,TASK-20260908T062225060390Z,응급키트 1번 만들어줘,"{'kit_type': '응급키트1번', 'items': [{'name': '수세미...",True,[],2026-09-08 06:22:45.150
1,TASK-20260908T062653244581Z,응급키트 1번 만들어줘,"{'kit_type': '응급키트1번', 'items': [{'name': '수세미...",True,[],2026-09-08 06:27:06.461
2,TASK-20260908T063104191011Z,응급키트 1번 만들어줘,"{'kit_type': '응급키트1번', 'items': [{'name': '샴푸리...",True,[],2026-09-08 06:31:14.846
3,TASK-20260908T063535978796Z,응급키트 1번 만들어,"{'kit_type': '응급키트1번', 'items': [{'name': '샴푸리...",True,[],2026-09-08 06:36:08.991
4,TASK-20260908T063921355753Z,응급키트 1번 만들어,"{'kit_type': '응급키트1번', 'items': [{'name': '샴푸리...",True,[],2026-09-08 06:39:49.896
5,TASK-20260908T064526000816Z,엘로우 로키 응급키트 2번 만들어,"{'kit_type': '응급키트2번', 'items': [{'name': '마스크...",True,[],2026-09-08 06:45:59.684
6,TASK-20260908T065051667419Z,응급키트 2번 만들어,"{'kit_type': '응급키트2번', 'items': [{'name': '마스크...",True,[],2026-09-08 06:51:16.247
7,TASK-20260908T065842478845Z,응급기트 2번 만들어,"{'kit_type': '응급키트2번', 'items': [{'name': '마스크...",True,[],2026-09-08 06:58:59.374
8,TASK-20260908T070429730427Z,응급키트 2번 만들어,"{'kit_type': '응급키트2번', 'items': [{'name': '마스크...",True,[],2026-09-08 07:04:44.528
9,TASK-20260908T070930692553Z,응급키트 2번 만들어,"{'kit_type': '응급키트2번', 'items': [{'name': '마스크...",True,[],2026-09-08 07:09:54.180


In [92]:
successful_command_count = len(command_eval)
valid_json_count = int(command_eval["json_valid"].sum())

json_conversion_rate = (
    valid_json_count / successful_command_count * 100
    if successful_command_count else 0
)

print(f"명령 변환 성공 항목: {successful_command_count}")
print(f"유효한 JSON 항목: {valid_json_count}")
print(f"명령 변환 성공률: {json_conversion_rate:.2f}%")


명령 변환 성공 항목: 22
유효한 JSON 항목: 22
명령 변환 성공률: 100.00%


### 파지: 클래스별 파지 성공률

In [93]:
component_docs = list(
    db.component_executions.find(
        {"task_id": {"$in": selected["task_id"].tolist()}},
        {
            "_id": 0,
            "task_id": 1,
            "component_index": 1,
            "class_name": 1,
            "slot": 1,
            "status": 1,
            "attempt_count": 1,
            "attempts": 1,
            "error": 1,
            "started_at": 1,
            "ended_at": 1,
        },
    )
)


In [94]:
grasp_review = pd.DataFrame([
    {
        "task_id": doc.get("task_id"),
        "component_index": doc.get("component_index"),
        "class_name": doc.get("class_name"),
        "slot": doc.get("slot"),
        "db_status": doc.get("status"),
        "attempt_count": doc.get("attempt_count"),
        "attempts": doc.get("attempts"),
        "error_code": (doc.get("error") or {}).get("code"),
        "started_at": doc.get("started_at"),
        "ended_at": doc.get("ended_at"),

        # 실제 물체 이동 여부를 보고 직접 입력
        "actual_grasp_success": pd.NA,
        "review_note": "",
    }
    for doc in component_docs
    if doc.get("status") != "SKIPPED"
]).sort_values(
    ["class_name", "started_at"]
).reset_index(drop=True)

grasp_review


,task_id,component_index,class_name,slot,db_status,attempt_count,attempts,error_code,started_at,ended_at,actual_grasp_success,review_note
0,TASK-20260908T064526000816Z,0,마스크,slot_1,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 06:45:59.787,2026-09-08 06:46:50.301,<NA>,
1,TASK-20260908T065051667419Z,0,마스크,slot_1,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 06:51:16.353,2026-09-08 06:52:06.807,<NA>,
2,TASK-20260908T065842478845Z,0,마스크,slot_1,FAILED,1,"[{'attempt_no': 1, 'status': 'FAILED', 'starte...",place_failed,2026-09-08 06:58:59.476,2026-09-08 06:59:41.348,<NA>,
3,TASK-20260908T070429730427Z,0,마스크,slot_1,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 07:04:44.630,2026-09-08 07:05:33.415,<NA>,
4,TASK-20260908T070930692553Z,0,마스크,slot_1,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 07:09:54.284,2026-09-08 07:10:41.922,<NA>,
...,...,...,...,...,...,...,...,...,...,...,...,...
111,TASK-20260908T075025330164Z,5,햄,slot_6,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 07:54:31.827,2026-09-08 07:55:14.624,<NA>,
112,TASK-20260908T075614673810Z,5,햄,slot_6,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 08:00:27.828,2026-09-08 08:01:08.027,<NA>,
113,TASK-20260908T080208052580Z,5,햄,slot_6,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 08:06:14.933,2026-09-08 08:06:59.129,<NA>,
114,TASK-20260908T080759225001Z,5,햄,slot_6,SUCCESS,1,"[{'attempt_no': 1, 'status': 'SUCCESS', 'start...",NaN,2026-09-08 08:12:11.234,2026-09-08 08:12:51.132,<NA>,


In [95]:
grasp_review["class_name"].value_counts()

class_name
양갱        20
수세미       16
컵라면       15
햄         15
분유        14
일회용숟가락    14
여행용티슈      9
샴푸리필       7
마스크        6
Name: count, dtype: int64

In [96]:
from pathlib import Path

# 생성 함수
# output_dir = Path("/home/shbong/Project2-ROS2-VLA/analyze")
# output_dir.mkdir(parents=True, exist_ok=False)

# grasp_review.to_csv(
#     output_dir / "grasp_review.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

In [97]:
csv_path = Path(
    "/home/shbong/Project2-ROS2-VLA/analyze/grasp_review.csv"
)

grasp_review = pd.read_csv(
    csv_path,
    encoding="utf-8-sig",
)

def parse_bool(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if value in {"true", "1", "yes", "y"}:
        return True
    if value in {"false", "0", "no", "n"}:
        return False

    return pd.NA

grasp_review["actual_grasp_success"] = (
    grasp_review["actual_grasp_success"].apply(parse_bool)
)

grasp_review["actual_grasp_success"].value_counts(dropna=False)

actual_grasp_success
True     84
False    32
Name: count, dtype: int64

#### 4. 클래스별 실제 파지 성공률

In [98]:
reviewed = grasp_review.dropna(
    subset=["actual_grasp_success"]
).copy()

class_grasp_summary = (
    reviewed
    .groupby("class_name")
    .agg(
        trials=("actual_grasp_success", "size"),
        success=("actual_grasp_success", "sum"),
    )
    .reset_index()
)

class_grasp_summary["success"] = (
    class_grasp_summary["success"].astype(int)
)

class_grasp_summary["grasp_success_rate_pct"] = (
    class_grasp_summary["success"]
    / class_grasp_summary["trials"]
    * 100
).round(2)

class_grasp_summary.sort_values("class_name")


,class_name,trials,success,grasp_success_rate_pct
0,마스크,6,1,16.67
1,분유,14,11,78.57
2,샴푸리필,7,6,85.71
3,수세미,16,13,81.25
4,양갱,20,14,70.00
5,여행용티슈,9,6,66.67
6,일회용숟가락,14,12,85.71
7,컵라면,15,8,53.33
8,햄,15,13,86.67


In [99]:
# 잘못된 파지 성공 판정 이유
reviewed["db_success"] = (
    reviewed["db_status"] == "SUCCESS"
)

reviewed["status_mismatch"] = (
    reviewed["db_success"]
    != reviewed["actual_grasp_success"]
)

reviewed.loc[
    reviewed["status_mismatch"],
    [
        "task_id",
        "component_index",
        "class_name",
        "db_status",
        "actual_grasp_success",
        "review_note",
    ],
]

,task_id,component_index,class_name,db_status,actual_grasp_success,review_note
0,TASK-20260908T064526000816Z,0,마스크,SUCCESS,False,NaN
1,TASK-20260908T065051667419Z,0,마스크,SUCCESS,False,NaN
3,TASK-20260908T070429730427Z,0,마스크,SUCCESS,False,NaN
5,TASK-20260908T071342411900Z,0,마스크,SUCCESS,False,NaN
8,TASK-20260908T065051667419Z,1,분유,FAILED,True,특이점도달
10,TASK-20260908T070429730427Z,2,분유,SUCCESS,False,NaN
17,TASK-20260908T073206626754Z,0,분유,SUCCESS,False,파지실패
38,TASK-20260908T075614673810Z,0,수세미,SUCCESS,False,잘못된객체파지
42,TASK-20260908T081523653769Z,0,수세미,SUCCESS,False,파지실패
43,TASK-20260908T062225060390Z,1,양갱,SUCCESS,False,NaN


### 시스템: 전체 공정 20회 기준 End-to-End 성공률

#### 키팅 완료 2건 - 검사 실패 / 키팅 성공

#### 클래스별 성공률

#### 품목 작업 성공률

#### 누락률

#### 검사 수행률

#### 검사 판정률

#### 구성 완료율


In [100]:
task_ids = selected["task_id"].tolist()

task_docs = list(
    db.kit_executions.find(
        {"task_id": {"$in": task_ids}},
        {"_id": 0}
    )
)

task_rows = []

for doc in task_docs:
    inspection = doc.get("final_inspection") or {}
    expected = inspection.get("expected_counts") or {}
    actual = inspection.get("actual_counts") or {}

    expected_total = sum(
        value for value in expected.values()
        if isinstance(value, (int, float))
    )

    actual_total = sum(
        value for value in actual.values()
        if isinstance(value, (int, float))
    )

    missing_total = sum(
        max(
            expected.get(name, 0) - actual.get(name, 0),
            0,
        )
        for name in expected
    )

    task_rows.append({
        "task_id": doc.get("task_id"),
        "task_status_db": doc.get("status"),
        "component_total": doc.get("component_total"),
        "inspection_result": inspection.get("result"),
        "inspection_performed": bool(inspection),
        "inspection_judged": (
            inspection.get("result") in {"PASS", "FAIL"}
        ),
        "expected_item_count": expected_total,
        "actual_item_count": actual_total,
        "missing_item_count": missing_total,
        "unexpected": inspection.get("unexpected") or [],
    })
task_eval = pd.DataFrame(task_rows)


In [101]:
# actual grasp aggregation
reviewed_grasps = grasp_review.dropna(subset=["actual_grasp_success"]).copy()
task_grasp = (reviewed_grasps.groupby("task_id").agg(
    attempted_components=("actual_grasp_success", "size"),
    successful_components=("actual_grasp_success", "sum"),
).reset_index())
for column in ["attempted_components", "successful_components", "kitting_completed"]:
    if column in task_eval.columns: task_eval = task_eval.drop(columns=column)
task_eval = task_eval.merge(task_grasp, on="task_id", how="left")
task_eval["component_total"] = pd.to_numeric(task_eval["component_total"], errors="coerce")
task_eval["attempted_components"] = task_eval["attempted_components"].fillna(0)
task_eval["successful_components"] = task_eval["successful_components"].fillna(0)
task_eval["kitting_completed"] = (task_eval["component_total"].gt(0) & task_eval["attempted_components"].ge(task_eval["component_total"]) & task_eval["successful_components"].ge(task_eval["component_total"]))
task_eval


,task_id,task_status_db,component_total,inspection_result,inspection_performed,inspection_judged,expected_item_count,actual_item_count,missing_item_count,unexpected,attempted_components,successful_components,kitting_completed
0,TASK-20260908T062225060390Z,FAILED,5,FAIL,True,True,5,1,4,[],5,3,False
1,TASK-20260908T062653244581Z,FAILED,5,FAIL,True,True,5,3,2,[],5,4,False
2,TASK-20260908T063104191011Z,FAILED,6,FAIL,True,True,6,4,2,[],6,5,False
3,TASK-20260908T063535978796Z,FAILED,6,ERROR,True,False,6,0,6,[],6,3,False
4,TASK-20260908T063921355753Z,FAILED,6,FAIL,True,True,6,3,3,[],6,3,False
5,TASK-20260908T064526000816Z,FAILED,6,FAIL,True,True,6,5,1,[],6,5,False
6,TASK-20260908T065051667419Z,FAILED,6,NaN,False,False,0,0,0,[],2,1,False
7,TASK-20260908T065842478845Z,FAILED,6,NaN,False,False,0,0,0,[],1,0,False
8,TASK-20260908T070429730427Z,FAILED,6,FAIL,True,True,6,6,1,[],6,3,False
9,TASK-20260908T070930692553Z,FAILED,6,FAIL,True,True,6,5,1,[],6,4,False


In [102]:
# metrics
EXPECTED_TRIALS = 20
TOTAL_TRIALS = len(selected)
if TOTAL_TRIALS != EXPECTED_TRIALS:
    print(f"주의: selected가 {TOTAL_TRIALS}건입니다. 정확한 20회 평가 대상을 확정하세요.")
kitting_completed_count = int(task_eval["kitting_completed"].sum())
kitting_completion_rate = kitting_completed_count / TOTAL_TRIALS * 100 if TOTAL_TRIALS else 0
attempted = int(reviewed_grasps["actual_grasp_success"].count())
grasp_success = int(reviewed_grasps["actual_grasp_success"].sum())
item_work_success_rate = grasp_success / attempted * 100 if attempted else 0
expected_items = task_eval["expected_item_count"].sum()
missing_items = task_eval["missing_item_count"].sum()
missing_rate = missing_items / expected_items * 100 if expected_items else 0
inspection_performed_count = int(task_eval["inspection_performed"].sum())
inspection_execution_rate = inspection_performed_count / TOTAL_TRIALS * 100 if TOTAL_TRIALS else 0
judged_count = int(task_eval["inspection_judged"].sum())
inspection_judgment_rate = judged_count / inspection_performed_count * 100 if inspection_performed_count else 0
composition_completion_rate = task_eval["actual_item_count"].sum() / expected_items * 100 if expected_items else 0
task_eval["e2e_success"] = task_eval["kitting_completed"] & task_eval["inspection_result"].eq("PASS")
e2e_success_count = int(task_eval["e2e_success"].sum())
kitting_completed_inspection_fail_count = int((task_eval["kitting_completed"] & task_eval["inspection_result"].eq("FAIL")).sum())
kitting_completed_inspection_pass_count = int((task_eval["kitting_completed"] & task_eval["inspection_result"].eq("PASS")).sum())


주의: selected가 22건입니다. 정확한 20회 평가 대상을 확정하세요.


In [103]:
print(f"End-to-End 성공률: {e2e_success_count}/{TOTAL_TRIALS} ({e2e_success_count / TOTAL_TRIALS * 100:.2f}%)" if TOTAL_TRIALS else "End-to-End 성공률: 0/0 (N/A)")
print(f"키팅 완료: {kitting_completed_count}/{TOTAL_TRIALS} ({kitting_completion_rate:.2f}%)" if TOTAL_TRIALS else "키팅 완료: 0/0 (N/A)")
print(f"키팅 완료 후 검사 FAIL: {kitting_completed_inspection_fail_count}건")
print(f"키팅 완료 후 검사 PASS: {kitting_completed_inspection_pass_count}건")
print(f"품목 작업 성공률: {grasp_success}/{attempted} ({item_work_success_rate:.2f}%)")
print(f"누락률: {missing_items}/{expected_items} ({missing_rate:.2f}%)")
print(f"검사 수행률: {inspection_performed_count}/{TOTAL_TRIALS} ({inspection_execution_rate:.2f}%)")
print(f"검사 판정률: {judged_count}/{inspection_performed_count} ({inspection_judgment_rate:.2f}%)")
print(f"구성 완료율: {task_eval['actual_item_count'].sum():.0f}/{expected_items:.0f} ({composition_completion_rate:.2f}%)")


End-to-End 성공률: 0/22 (0.00%)
키팅 완료: 3/22 (13.64%)
키팅 완료 후 검사 FAIL: 3건
키팅 완료 후 검사 PASS: 0건
품목 작업 성공률: 84/116 (72.41%)
누락률: 38/112 (33.93%)
검사 수행률: 19/22 (86.36%)
검사 판정률: 18/19 (94.74%)
구성 완료율: 78/112 (69.64%)


In [104]:
task_eval["inspection_result"].value_counts(dropna=False)


inspection_result
FAIL     18
NaN       3
ERROR     1
Name: count, dtype: int64

In [105]:
# 안정성: 클래스별 연속 실제 파지 성공 횟수
stability_input = grasp_review.dropna(
    subset=["actual_grasp_success", "ended_at"]
).copy()

stability_input["ended_at"] = pd.to_datetime(
    stability_input["ended_at"],
    errors="coerce",
)
stability_input = stability_input.dropna(subset=["ended_at"])

def consecutive_success_stats(group):
    group = group.sort_values(
        ["ended_at", "task_id", "component_index"]
    )

    longest = 0
    current = 0

    for success in group["actual_grasp_success"].astype(bool):
        if success:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    trailing = 0
    for success in reversed(
        group["actual_grasp_success"].astype(bool).tolist()
    ):
        if not success:
            break
        trailing += 1

    return pd.Series({
        "reviewed_trials": len(group),
        "success_count": int(
            group["actual_grasp_success"].sum()
        ),
        "longest_consecutive_successes": longest,
        "current_trailing_successes": trailing,
    })

class_stability = (
    stability_input
    .groupby("class_name", dropna=False)
    .apply(consecutive_success_stats)
    .reset_index()
)

class_stability["success_rate_pct"] = (
    class_stability["success_count"]
    / class_stability["reviewed_trials"]
    * 100
).round(2)

class_stability


,class_name,reviewed_trials,success_count,longest_consecutive_successes,current_trailing_successes,success_rate_pct
0,마스크,6,1,1,0,16.67
1,분유,14,11,4,2,78.57
2,샴푸리필,7,6,6,0,85.71
3,수세미,16,13,11,0,81.25
4,양갱,20,14,7,7,70.00
5,여행용티슈,9,6,6,0,66.67
6,일회용숟가락,14,12,9,2,85.71
7,컵라면,15,8,3,0,53.33
8,햄,15,13,10,10,86.67
